# 第９回　AIによる臓器の自動抽出（医療画像処理工学演習）

このノートブックでは，深層学習（ディープラーニング）を用いた臓器の自動抽出ソフトウェア **TotalSegmentator** を使って，胸部CT画像から肺や心臓などの領域を抽出します．

**使い方**：上のセルから順に，セルの左にある ▶ を押してください．コードを書き換える必要はありません．
各セルの実行が終わると，▶ の左に緑色のチェックが付きます．赤い文字の警告が出ても，最後に「できました」と表示されれば問題ありません．

**うまくいかないとき**：メニューの「ランタイム」→「ランタイムを接続解除して削除」を選び，１．からやり直してください．

---
使用する画像：LIDC-IDRI（LIDC-IDRI-0265），The Cancer Imaging Archive，CC BY 3.0（https://doi.org/10.7937/K9/TCIA.2015.LO9QL9SX）<br>
引用文献などの詳細は，リポジトリの README（https://github.com/ichikawalab/practice-medical-imaging）を参照してください．<br>
使用するソフトウェア：TotalSegmentator（Wasserthal J, et al. Radiology: Artificial Intelligence. 2023;5(5):e230024）<br>
＊TotalSegmentatorは研究用のソフトウェアであり，医療機器ではありません．

In [ ]:
#@title １．GPU（AIの計算に使う装置）が使えるか確認する
import torch
if torch.cuda.is_available():
    print('GPUが使えます：', torch.cuda.get_device_name(0))
    print('できました．２．に進んでください．')
else:
    print('GPUが使えません．')
    print('メニューの「ランタイム」→「ランタイムのタイプを変更」で「T4 GPU」を選んで保存し，このセルをもう一度実行してください．')
    print('（GPUが割り当てられない場合は，そのまま進めても動きますが，４．に10分以上かかります．）')

In [ ]:
#@title ２．TotalSegmentatorを準備する（数分かかります）
!pip install -q TotalSegmentator tifffile
print('できました．３．に進んでください．')

In [ ]:
#@title ３．CT画像を読み込む
import os, glob, zipfile, urllib.request
import numpy as np, pydicom, nibabel as nib

URL = 'https://github.com/ichikawalab/practice-medical-imaging/raw/main/imgproc/data/LIDC-IDRI-0265_CT.zip'
if not os.path.isdir('LIDC-IDRI-0265_CT'):
    urllib.request.urlretrieve(URL, 'LIDC-IDRI-0265_CT.zip')
    zipfile.ZipFile('LIDC-IDRI-0265_CT.zip').extractall('.')

# DICOMを頭側から並べ，CT値（HU）の3次元配列 ct[スライス, 行, 列] にする（ImageJの「Image Sequence」と同じ並び）
ds = [pydicom.dcmread(f) for f in glob.glob('LIDC-IDRI-0265_CT/*.dcm')]
ds.sort(key=lambda d: -float(d.ImagePositionPatient[2]))
ct = np.stack([d.pixel_array * float(d.RescaleSlope) + float(d.RescaleIntercept) for d in ds]).astype(np.int16)
dy, dx = (float(v) for v in ds[0].PixelSpacing)
p0, p1 = (np.array(d.ImagePositionPatient, float) for d in ds[:2])
dz = float(np.linalg.norm(p1 - p0))

# NIfTI形式（TotalSegmentatorの入力形式）にする．DICOMの座標（LPS）をNIfTIの座標（RAS）に直す
r = np.array(ds[0].ImageOrientationPatient[:3], float)   # 列方向（画像の右向き）
c = np.array(ds[0].ImageOrientationPatient[3:], float)   # 行方向（画像の下向き）
lps2ras = np.diag([-1, -1, 1])
affine = np.eye(4)
affine[:3, 0] = lps2ras @ r * dx
affine[:3, 1] = lps2ras @ c * dy
affine[:3, 2] = lps2ras @ (p1 - p0)
affine[:3, 3] = lps2ras @ p0
ct_img = nib.Nifti1Image(ct.transpose(2, 1, 0), affine)
ct_img.header.set_xyzt_units('mm')

print(f'画像の大きさ：{ct.shape[2]}×{ct.shape[1]}画素，{ct.shape[0]}枚')
print(f'ボクセルの大きさ：{dx:.2f} mm × {dy:.2f} mm × {dz:.2f} mm')
print('できました．４．に進んでください．')

In [ ]:
#@title ４．AI（TotalSegmentator）で臓器を自動抽出する（GPUで数分かかります）
from totalsegmentator.python_api import totalsegmentator
from totalsegmentator.map_to_binary import class_map

seg_img = totalsegmentator(ct_img, None, ml=True, fast=not torch.cuda.is_available(), quiet=True)
seg = np.asarray(seg_img.dataobj).transpose(2, 1, 0).astype(np.uint8)   # ctと同じ並び [スライス, 行, 列]
NAMES = class_map['total']
found = [NAMES[i] for i in np.unique(seg) if i]
print(f'{len(found)}種類の構造が抽出されました．')
print('できました．５．に進んでください．')

In [ ]:
#@title ５．抽出結果を見る（臓器を選んでから ▶ を押す）
臓器 = '肺（5つの肺葉をまとめたもの）'  #@param ['肺（5つの肺葉をまとめたもの）', '心臓', '大動脈', '気管', '食道', '肝臓', '椎骨（まとめたもの）', '肋骨（まとめたもの）']
import matplotlib.pyplot as plt
from IPython.display import display   # TotalSegmentatorを読み込むと plt.show() では図が出なくなるため，display() で表示する

ORGANS = {'肺（5つの肺葉をまとめたもの）': 'lung_', '心臓': 'heart', '大動脈': 'aorta', '気管': 'trachea', '食道': 'esophagus', '肝臓': 'liver', '椎骨（まとめたもの）': 'vertebrae_', '肋骨（まとめたもの）': 'rib_'}

def organ_mask(key):
    ids = [i for i, n in NAMES.items() if (n.startswith(key) if key.endswith('_') else n == key)]
    return np.isin(seg, ids)

mask = organ_mask(ORGANS[臓器])
if not mask.any():
    print('この画像では，選んだ臓器は抽出されませんでした．別の臓器を選んでください．')
z = int(np.argmax(mask.sum(axis=(1, 2))))   # 選んだ臓器が一番大きく写っているスライス
y = int(np.argmax(mask.sum(axis=(0, 2))))

def show(ax, img, m, aspect, title):
    ax.imshow(img, cmap='gray', vmin=-1350, vmax=150, aspect=aspect)   # 肺野条件（WL -600，WW 1500）
    ax.imshow(np.ma.masked_where(~m, m), cmap='autumn', alpha=0.4, aspect=aspect)
    ax.set_title(title); ax.axis('off')

fig, axs = plt.subplots(1, 2, figsize=(12, 6))
show(axs[0], ct[z], mask[z], 1, f'Axial (slice {z + 1})')   # 横断像（グラフの文字は英語で表示）
show(axs[1], ct[:, y, :], mask[:, y, :], dz / dx, f'Coronal MPR (row {y + 1})')   # 冠状断像
display(fig); plt.close(fig)
print(f'左：横断像（{z + 1}枚目），右：冠状断像（MPR，上から{y + 1}行目）．赤い部分がAIの抽出結果．')
print('ほかの臓器も見るときは，臓器を選び直して ▶ を押してください．')

In [ ]:
#@title ６．結果をImageJ用に保存してダウンロードする
import tifffile
from google.colab import files

def save_tif(name, arr):
    tifffile.imwrite(name, arr, imagej=True, resolution=(1 / dx, 1 / dy),
                     metadata={'spacing': dz, 'unit': 'mm', 'axes': 'ZYX'})

save_tif('AI_lung.tif', organ_mask('lung_').astype(np.uint8) * 255)
save_tif('AI_labels.tif', seg)
with open('AI_labels.txt', 'w', encoding='utf-8') as f:
    for i in sorted(set(np.unique(seg)) - {0}):
        f.write(f'{i}\t{NAMES[i]}\n')
with zipfile.ZipFile('AI_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for n in ['AI_lung.tif', 'AI_labels.tif', 'AI_labels.txt']:
        zf.write(n)
files.download('AI_results.zip')
print('できました．AI_results.zip がダウンロードされます．')
print('  AI_lung.tif    ：肺の領域（画素値255）の3次元マスク')
print('  AI_labels.tif  ：抽出されたすべての構造のラベル画像（画素値＝構造の番号）')
print('  AI_labels.txt  ：構造の番号と名前の対応表')